In [1]:
# staging_multi_stream_loader.py
# Streams weather, demand, price, and city_attributes into stg_ with full progress tracking

import pymysql
from pathlib import Path
import time
import getpass

BASE = Path(r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\energy_project_data")

DATASETS = {
    "weather": {
        "dir": BASE / "weather_chunks",
        "single_file": False,
        "target_table": "stg_weather_raw",
        "sql": """
LOAD DATA LOCAL INFILE '{path}'
INTO TABLE stg_weather_raw
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\\r\\n'
IGNORE 1 LINES
(@source_file,@measure,@datetime_utc,@city_name,@value_text,@value_num)
SET
  source_file  = @source_file,
  measure      = @measure,
  datetime_utc = NULLIF(TRIM(@datetime_utc),''),
  city_name    = @city_name,
  value_text   = NULLIF(REPLACE(@value_text, '\\r', ''), ''),
  value_num    = NULLIF(REPLACE(@value_num,  '\\r', ''), '');
"""
    },
    "demand": {
        "dir": BASE / "demand_chunks",
        "single_file": False,
        "target_table": "stg_demand",
        "sql": """
LOAD DATA LOCAL INFILE '{path}'
INTO TABLE stg_demand
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\\r\\n'
IGNORE 1 LINES
(MeasureItem, DateUTC, DateShort, TimeFrom, TimeTo, CountryCode,
 Cov_ratio, Value, Value_ScaleTo100, year);
"""
    },
    "price": {
        "dir": BASE / "price_chunks",
        "single_file": False,
        "target_table": "stg_price",
        "sql": """
LOAD DATA LOCAL INFILE '{path}'
INTO TABLE stg_price
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\\r\\n'
IGNORE 1 LINES
(Country, ISO3_Code, Datetime_UTC, Datetime_Local, Price_EUR_MWhe);
"""
    },
    "city_attributes": {
        "dir": BASE,
        "single_file": True,
        "file": BASE / "city_attributes.csv",
        "target_table": "stg_city_attributes",
        "sql": """
LOAD DATA LOCAL INFILE '{path}'
INTO TABLE stg_city_attributes
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\\r\\n'
IGNORE 1 LINES
(City, Country, Latitude, Longitude);
"""
    }
}


def log(msg: str) -> None:
    now = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{now}] {msg}")


def recreate_staging_tables(conn):
    log("Recreating staging tables stg_* ...")
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS stg_demand;")
        cur.execute("""
            CREATE TABLE IF NOT EXISTS stg_demand (
                raw_id INT AUTO_INCREMENT PRIMARY KEY,
                MeasureItem VARCHAR(255),
                DateUTC VARCHAR(64),
                DateShort VARCHAR(32),
                TimeFrom VARCHAR(64),
                TimeTo VARCHAR(64),
                CountryCode VARCHAR(8),
                Cov_ratio DECIMAL(8,4),
                Value DECIMAL(12,4),
                Value_ScaleTo100 DECIMAL(12,4),
                year INT,
                loaded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
        """)

        cur.execute("DROP TABLE IF EXISTS stg_price;")
        cur.execute("""
            CREATE TABLE IF NOT EXISTS stg_price (
                raw_id INT AUTO_INCREMENT PRIMARY KEY,
                Country VARCHAR(128),
                ISO3_Code VARCHAR(8),
                Datetime_UTC VARCHAR(64),
                Datetime_Local VARCHAR(64),
                Price_EUR_MWhe DECIMAL(10,4),
                loaded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
        """)

        cur.execute("DROP TABLE IF EXISTS stg_city_attributes;")
        cur.execute("""
            CREATE TABLE IF NOT EXISTS stg_city_attributes (
                raw_id INT AUTO_INCREMENT PRIMARY KEY,
                City VARCHAR(128),
                Country VARCHAR(128),
                Latitude DECIMAL(10,6),
                Longitude DECIMAL(10,6),
                loaded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
        """)

        cur.execute("DROP TABLE IF EXISTS stg_weather_raw;")
        cur.execute("""
            CREATE TABLE IF NOT EXISTS stg_weather_raw (
                raw_id INT AUTO_INCREMENT PRIMARY KEY,
                source_file VARCHAR(255),
                measure VARCHAR(128),
                datetime_utc DATETIME NULL,
                city_name VARCHAR(128),
                value_text VARCHAR(255),
                value_num DECIMAL(12,4),
                loaded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
        """)

    conn.commit()
    log("Staging tables recreated.")


def load_file_with_report(conn, dataset_label, file_path, sql_template):
    start = time.time()
    file_size_mb = round(file_path.stat().st_size / 1024 / 1024, 2)

    print(f"\n=== FILE LOAD REPORT ({dataset_label.upper()}) ===")
    print(f"File Name     : {file_path.name}")
    print(f"Full Path     : {file_path}")
    print(f"File Size     : {file_size_mb} MB")

    rows = 0
    status = "SUCCESS"
    error_msg = None

    try:
        sql = sql_template.format(path=file_path.as_posix())
        with conn.cursor() as cur:
            cur.execute(sql)
            rows = cur.rowcount if cur.rowcount is not None else 0
    except Exception as e:
        status = "ERROR"
        error_msg = str(e)

    duration = round(time.time() - start, 2)
    throughput = round(rows / duration, 2) if duration > 0 else rows

    print(f"Rows Loaded   : {rows}")
    print(f"Duration      : {duration}s")
    print(f"Throughput    : {throughput} rows/sec")
    print(f"Status        : {status}")
    if error_msg:
        print(f"Error         : {error_msg}")
    print(f"==============================")

    return rows, duration, status


def main():
    password = getpass.getpass("Enter your MySQL password: ")

    conn = pymysql.connect(
        host="localhost",
        user="Oyeniyi_ETL",
        password=password,
        database="electricity_capstone",
        local_infile=1,
        autocommit=True
    )

    try:
        recreate_staging_tables(conn)

        for label, cfg in DATASETS.items():
            total_rows = 0
            total_duration = 0.0
            file_count = 0

            # Single-file dataset
            if cfg.get("single_file", False):
                file_path = cfg["file"]
                if not file_path.exists():
                    print(f"\n[SKIP] {label.upper()} file not found: {file_path}")
                    continue

                print(f"\n=== LOADING DATASET: {label.upper()} (1 file) ===")
                rows, duration, _ = load_file_with_report(conn, label, file_path, cfg["sql"])
                total_rows += rows
                total_duration += duration
                file_count = 1

            else:
                data_dir = cfg["dir"]
                if not data_dir.exists():
                    print(f"\n[SKIP] {label.upper()} directory not found: {data_dir}")
                    continue

                chunks = sorted(data_dir.glob("*.csv"))
                if not chunks:
                    print(f"\n[SKIP] No chunks found for {label.upper()} in {data_dir}")
                    continue

                print(f"\n=== LOADING DATASET: {label.upper()} ({len(chunks)} files) ===")
                for i, chunk in enumerate(chunks, start=1):
                    print(f"\n--- [{label.upper()}] File {i}/{len(chunks)} ---")
                    rows, duration, _ = load_file_with_report(conn, label, chunk, cfg["sql"])
                    total_rows += rows
                    total_duration += duration
                    file_count += 1

            # Dataset summary + table row count
            if file_count > 0:
                avg_throughput = round(total_rows / total_duration, 2) if total_duration > 0 else total_rows
                target_table = cfg["target_table"]
                with conn.cursor() as cur:
                    cur.execute(f"SELECT COUNT(*) FROM {target_table};")
                    table_rows = cur.fetchone()[0] or 0

                print(f"\n=== DATASET SUMMARY: {label.upper()} ===")
                print(f"Files Loaded  : {file_count}")
                print(f"Total Rows    : {total_rows}")
                print(f"Total Time    : {round(total_duration, 2)}s")
                print(f"Avg Throughput: {avg_throughput} rows/sec")
                print(f"Rows in {target_table}: {table_rows}")
                print(f"========================================")

    finally:
        conn.close()
        log("Connection closed.")


if __name__ == "__main__":
    main()


Enter your password:  ········
Enter your MySQL password:  ········



=== LOADING DATASET: WEATHER (24 files) ===

=== FILE LOAD REPORT (WEATHER) ===
File Name     : unpivot_humidity.part1.csv
Full Path     : C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\energy_project_data\weather_chunks\unpivot_humidity.part1.csv
File Size     : 30.07 MB
Rows Loaded   : 500000
Duration      : 21.56s
Throughput    : 23191.09 rows/sec
Status        : SUCCESS

=== FILE LOAD REPORT (WEATHER) ===
File Name     : unpivot_humidity.part2.csv
Full Path     : C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\energy_project_data\weather_chunks\unpivot_humidity.part2.csv
File Size     : 30.08 MB
Rows Loaded   : 500000
Duration      : 25.69s
Throughput    : 19462.83 rows/sec
Status        : SUCCESS

=== FILE LOAD REPORT (WEATHER) ===
File Name     : unpivot_humidity.part3.csv
Full Path     : C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\energy_project_data\weather_chunks\unpivot_humidity.part3.csv
File Size     : 29.86 MB
Rows Loaded   : 500000
Duration      : 22.07s
Throughput    : 2